In [9]:
%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np
from IPython.display import HTML
from scipy import linalg as LA
import random
import numpy.matlib
import kwant
import tinyarray
import multiprocessing as mp
import os
from tqdm import tqdm
import helpers as hp
from pathlib import Path
from config import PathConfigs
import scipy.sparse.linalg as sla
import multiprocessing as mp
from functools import partial
from scipy.signal import find_peaks
from scipy.spatial import KDTree

import skfda
from skfda.exploratory.visualization import FPCAPlot
from skfda.preprocessing.dim_reduction import FPCA
from skfda.representation.basis import (
    BSplineBasis,
    FourierBasis,
    MonomialBasis,
)


from IPython.display import display, HTML
from main_parallel import worker_pdi_step, worker_simulation_step
display(HTML('<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'))



#pauli matrices
sigma_0 = tinyarray.array([[1, 0], [0, 1]])
sigma_x = tinyarray.array([[0, 1], [1, 0]])
sigma_y = tinyarray.array([[0, -1j], [1j, 0]])
sigma_z = tinyarray.array([[1, 0], [0, -1]])

In [10]:

save_plots = True
#dirname = Path(PathConfigs.DATA/"dis_realizations/disorder_realization_6_results")
dirname = Path(PathConfigs.DATA/"Tdis_pfaff3")

plot_dir = Path(dirname, "Plots")
os.makedirs(plot_dir, exist_ok=True)


In [11]:


os.makedirs(Path(dirname), exist_ok=True)

#params = np.load(Path(dir/"all_params.npz"))
params = np.load(Path(dirname/"all_params.npz"), allow_pickle=True)
mu_n = float(params['mu_n'])
t = float(params['t'])
mu_leads =float(params['mu_leads'])
Delta0 = float(params['Delta0'])
gamma = float(params['gamma'])
alpha = float(params['alpha'])
Ln = int(params['Ln']) # normal metal length
Lb = int(params['Lb']) #barrier length
Ls = int(params['Ls']) #super conductor length
barrier_l = float(params['barrier0'])
V0 = float(params['V0'])
points = 100#int(params['Upoints'])

totlen = Ln + Lb +Ls 

paramd = {name:params[name] for name in params.files}
del paramd['Vdisx']
del paramd['energies']
del paramd['barrier_arr']
del paramd['mu_var']


pdicalc = hp.PDICalculator(t, alpha, gamma, Ls, params['Vdisx'], V0)

In [12]:

barrier_left_conductance_left_arr = hp.np_load_wrapped("barrier_left_conductance_left_arr", dirname)
barrier_right_conductance_right_arr = hp.np_load_wrapped("barrier_right_conductance_right_arr", dirname)
barrier_left_conductance_right_arr = hp.np_load_wrapped("barrier_left_conductance_right_arr", dirname)
barrier_right_conductance_left_arr = hp.np_load_wrapped("barrier_right_conductance_left_arr", dirname)
dIdVs_left_arr = hp.np_load_wrapped("dIdVs_left_arr", dirname)
dIdVs_right_arr = hp.np_load_wrapped("dIdVs_right_arr", dirname)
energies = hp.np_load_wrapped("energies", dirname)
pdi_arr = hp.np_load_wrapped("pdi_data", dirname)
#Conductance_matrix = hp.np_load_wrapped("Conductance_matrix_zero_energy", dirname)
#gamma_sq_arr = hp.np_load_wrapped("gamma_sq_arr", dirname)
#mp_arr = hp.np_load_wrapped("mp_arr", dirname)
rG_corr = hp.np_load_wrapped("rG_corr", dirname)
params_list = np.load(Path(dirname, 'params_list.npy'))
#spectrum_arr = hp.np_load_wrapped("spectrum_arr", dirname)
Vdisx = hp.np_load_wrapped("Vdisx", dirname)
barrier_arr = hp.np_load_wrapped("barrier_arr", dirname)
peaks_left = hp.np_load_wrapped("peaks_left", dirname)
peaks_right = hp.np_load_wrapped("peaks_right", dirname)
site_localizations = 1 - (hp.np_load_wrapped("site_localizations", dirname)/300)
weight_localizations = hp.np_load_wrapped("weight_localization_arr", dirname)
overlaps = hp.np_load_wrapped("OverlapIntegral", dirname)
mzm_seps = hp.np_load_wrapped("mzm_separation_arr", dirname)

Tgap = hp.np_load_wrapped("topological_gap", dirname)

Vdisx = params['Vdisx']  * V0


#ldos = hp.np_load_wrapped("LDOS", dirname)

#return [mu_pm * V_c, vz_raw, pdi_val]
#new_pdi_dat = np.asarray([[pdr[0]/V_c, pdr[1]*V_c, pdr[2]] for pdr in pdi_arr])




In [13]:
barrier_right_conductance_right_arr

array([[5.11601981e-04, 8.62858139e-05, 1.96182925e-05, ...,
        5.41790184e-12, 3.90686153e-12, 2.84215323e-12],
       [5.11918609e-04, 8.63361478e-05, 1.96293284e-05, ...,
        5.42124782e-12, 3.90894258e-12, 2.84330303e-12],
       [5.12877719e-04, 8.64887218e-05, 1.96627964e-05, ...,
        5.42911653e-12, 3.91500062e-12, 2.84722339e-12],
       ...,
       [3.27405300e-01, 1.17940447e-01, 4.67539592e-02, ...,
        1.78648820e-05, 1.51947143e-05, 1.29798097e-05],
       [6.07387864e-01, 2.20255883e-01, 8.64157253e-02, ...,
        1.86133139e-05, 1.57139440e-05, 1.33298407e-05],
       [8.56575216e-01, 2.59310640e-01, 9.26254204e-02, ...,
        2.22215158e-05, 1.88214687e-05, 1.60159695e-05]])